# Лабораторная работа №2 — Обработка и подготовка данных в Pandas

**Студент:** Фадеев А. М.  **Группа:** 6401

**Датасет:** Poker Tournament Survivorship — https://www.kaggle.com/datasets/sergionefedov/poker-tournament-survivorship

45 000 игроков в турнирный покер, 17 признаков: шкалы подготовки и дисциплины (`skill`, `bankroll_mgmt`, `tilt_control`, `study_hours_week`), параметры практики (`tournaments_played`, `avg_field_size`, `avg_buyin`, `bankroll_buyins`), результаты по дистанции (`itm_rate`, `final_table_rate`, `win_rate`, `realized_roi`) и три целевых признака: непрерывный `realized_roi`, бинарные `went_broke` и `profitable`.

Рабочая выгрузка `data/poker_raw.csv` — та же выборка, выгруженная «сырой»: разделитель `;`, пропуски закодированы токенами `unknown`, `no_data`, `-`, добавлены полные дубликаты строк. Мини-файлы для заданий 1.1–1.2 и 3.2: `stake_levels.xlsx`, `venues.json`, `series_schedule.csv`, `formats_cp1251.csv`.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

DATA_DIR = "data"
RAW_PATH = f"{DATA_DIR}/poker_raw.csv"
MISSING_TOKENS = ["unknown", "no_data", "-"]
TO_NUMERIC = ["skill", "tilt_control", "study_hours_week", "bankroll_buyins"]

print("pandas:", pd.__version__)

pandas: 2.2.3


## Занятие 1. Загрузка данных и обзор структуры

### Задание 1.1. Загрузка данных из CSV, Excel и JSON

In [2]:
df = pd.read_csv(RAW_PATH, sep=";")
stake_levels = pd.read_excel(f"{DATA_DIR}/stake_levels.xlsx")
venues = pd.read_json(f"{DATA_DIR}/venues.json")

print("read_csv:", df.shape)
print(df.head(3))
print()
print("read_excel:", stake_levels.shape)
print(stake_levels)
print()
print("read_json:", venues.shape)
print(venues)

read_csv: (45700, 17)
   player_id skill  bankroll_mgmt tilt_control study_hours_week  tournaments_played  avg_field_size  avg_buyin bankroll_buyins  itm_rate  final_table_rate  win_rate  biggest_cash_x  \
0  PLR044346  4.67           8.16         5.04              7.6                  18             554      16.71               6      10.0              0.00       0.0             4.9   
1  PLR004550  5.14           1.37         3.17              5.3                 118             909      66.35              41       4.3              0.00       0.0             6.8   
2  PLR026203   8.9           2.17         6.79              2.2                  93             559      41.99              62       5.6              2.78       0.0             7.4   

   final_bankroll_bi  realized_roi  went_broke  profitable  
0                0.1         -54.2           1           0  
1                0.2         -82.1           1           0  
2                0.3         -79.4           1           0

### Задание 1.2. Ключевые параметры `read_csv`

`sep` — разделитель полей: в файле это `;`, при разборе по умолчанию все 17 полей склеиваются в один столбец. `encoding` — кодировка: `formats_cp1251.csv` записан в cp1251 и при чтении как UTF-8 даёт `UnicodeDecodeError`. `header` — номер строки с заголовком: при `header=None` имена столбцов попадают в данные. `usecols` — набор читаемых столбцов, экономит память. `parse_dates` — приведение столбца к `datetime64` при чтении, иначе дата остаётся строкой и к ней неприменимы `.dt` и разность дат. `na_values` — дополнительные маркеры пропуска: без них числовые столбцы читаются как `object`.

In [3]:
wrong_sep = pd.read_csv(RAW_PATH, nrows=3)
print("sep по умолчанию:", wrong_sep.shape, "| sep=';':", pd.read_csv(RAW_PATH, sep=";", nrows=3).shape)
print()

try:
    pd.read_csv(f"{DATA_DIR}/formats_cp1251.csv")
except UnicodeDecodeError as err:
    print("encoding: UnicodeDecodeError ->", err)
print(pd.read_csv(f"{DATA_DIR}/formats_cp1251.csv", encoding="cp1251"))
print()

print("header=None:")
print(pd.read_csv(RAW_PATH, sep=";", header=None, nrows=2).iloc[:, :4])
print()

part = pd.read_csv(RAW_PATH, sep=";", usecols=["player_id", "realized_roi"])
print("usecols:", part.shape, round(part.memory_usage(deep=True).sum() / 1024 ** 2, 2), "МБ против",
      round(df.memory_usage(deep=True).sum() / 1024 ** 2, 2), "МБ")
print()

schedule = pd.read_csv(f"{DATA_DIR}/series_schedule.csv", parse_dates=["series_start"])
print("parse_dates: без параметра",
      pd.read_csv(f"{DATA_DIR}/series_schedule.csv")["series_start"].dtype,
      "| с параметром", schedule["series_start"].dtype)
print("разрыв между сериями, дней:", schedule["series_start"].diff().dt.days.dropna().tolist())
print()

df_na = pd.read_csv(RAW_PATH, sep=";", na_values=MISSING_TOKENS)
print("na_values:")
print(pd.DataFrame({
    "dtype_без": df[TO_NUMERIC].dtypes.astype(str),
    "пропусков_без": df[TO_NUMERIC].isnull().sum(),
    "dtype_с": df_na[TO_NUMERIC].dtypes.astype(str),
    "пропусков_с": df_na[TO_NUMERIC].isnull().sum(),
}))

sep по умолчанию: (3, 1) | sep=';': (3, 17)

encoding: UnicodeDecodeError -> 'utf-8' codec can't decode byte 0xf4 in position 0: invalid continuation byte
            формат                 описание  доля_событий
0          Фриролл               Без взноса          0.18
1         Сателлит  Отбор в главное событие          0.24
2  Основной турнир         Классический MTT          0.49
3        Хайроллер          Высокие бай-ины          0.09

header=None:
           0      1              2             3
0  player_id  skill  bankroll_mgmt  tilt_control
1  PLR044346   4.67           8.16          5.04



usecols: (45700, 2) 2.88 МБ против 15.82 МБ

parse_dates: без параметра object | с параметром datetime64[ns]
разрыв между сериями, дней: [98.0, 77.0, 105.0, 84.0]

na_values:
                 dtype_без  пропусков_без  dtype_с  пропусков_с
skill               object              0  float64          907
tilt_control        object              0  float64         1525
study_hours_week    object              0  float64         2663
bankroll_buyins     object              0  float64          546


### Задание 1.3. Общее описание датасета

In [4]:
print("head(3):")
print(df.head(3))
print()
print("tail(3):")
print(df.tail(3))
print()
print("sample(5):")
print(df.sample(5, random_state=42))
print()
print("shape:", df.shape)
df.info()
print()
print("dtypes:")
print(df.dtypes)
print()
print("describe():")
print(df.describe().T)
print()
print("describe(include='object'):")
print(df.describe(include="object").T)

head(3):
   player_id skill  bankroll_mgmt tilt_control study_hours_week  tournaments_played  avg_field_size  avg_buyin bankroll_buyins  itm_rate  final_table_rate  win_rate  biggest_cash_x  \
0  PLR044346  4.67           8.16         5.04              7.6                  18             554      16.71               6      10.0              0.00       0.0             4.9   
1  PLR004550  5.14           1.37         3.17              5.3                 118             909      66.35              41       4.3              0.00       0.0             6.8   
2  PLR026203   8.9           2.17         6.79              2.2                  93             559      41.99              62       5.6              2.78       0.0             7.4   

   final_bankroll_bi  realized_roi  went_broke  profitable  
0                0.1         -54.2           1           0  
1                0.2         -82.1           1           0  
2                0.3         -79.4           1           0  

tail(3):


### Задание 1.4. Характеристика столбцов и аномалии

Состав набора: 17 столбцов — один идентификатор, четыре шкалы подготовки и дисциплины, четыре параметра игровой практики, пять метрик результата и три целевых признака.

| Столбец | Тип признака | Что значит | Диапазон |
| --- | --- | --- | --- |
| `player_id` | категориальный (идентификатор) | уникальный номер игрока, ключ дедупликации и соединений | 45 000 значений |
| `skill` | количественный (непрерывный) | истинное преимущество игрока на длинной дистанции; за один турнир даёт почти ноль, накапливается объёмом | 0–10 |
| `bankroll_mgmt` | количественный (непрерывный) | дисциплина выбора лимита под размер банкролла | 0–10 |
| `tilt_control` | количественный (непрерывный) | контроль тильта: насколько игрок не сливает на эмоциях | 0–10 |
| `study_hours_week` | количественный (непрерывный) | часов разбора раздач и теории в неделю | 0–40 |
| `tournaments_played` | количественный (дискретный) | сколько турниров сыграно в выборке, то есть длина дистанции | 10–1 567 |
| `avg_field_size` | количественный (дискретный) | среднее число участников турнира; больше поле — сильнее смещение выплат в топ — выше дисперсия | 20–3 835 |
| `avg_buyin` | количественный (непрерывный) | средний бай-ин в долларах | 2.0–2 913.45 |
| `bankroll_buyins` | количественный (дискретный) | стартовый банкролл, измеренный в бай-инах; главный драйвер risk of ruin | 5–347 |
| `itm_rate` | количественный (доля, %) | доля турниров с попаданием в призы; близка к доле оплачиваемых мест, слабый сигнал мастерства | 0–60 |
| `final_table_rate` | количественный (доля, %) | доля выходов за финальный стол; сигнал мастерства лучше, чем ITM | 0–30 |
| `win_rate` | количественный (доля, %) | доля первых мест; мала даже у сильных игроков | 0–16.67 |
| `biggest_cash_x` | количественный (непрерывный) | крупнейшая разовая выплата в бай-инах, мера удачного глубокого забега | 0–200 |
| `final_bankroll_bi` | количественный (непрерывный) | банкролл на конец выборки в бай-инах, 0 если обанкротился | 0–527.6 |
| `realized_roi` | количественный (целевой, %) | целевой признак регрессии: фактический ROI за выборку, снизу ограничен −100% | −100–400 |
| `went_broke` | категориальный (бинарный, целевой) | целевой признак классификации: 1, если банкролл дошёл до нуля | 0 / 1 |
| `profitable` | категориальный (бинарный, целевой) | целевой признак классификации: 1, если ROI > 0 | 0 / 1 |

Порядковых признаков и столбцов с датой в исходном наборе нет: даты лежат в `series_schedule.csv`, порядковые признаки создаются в заданиях 2.2 и 2.5. Четыре количественных столбца (`skill`, `tilt_control`, `study_hours_week`, `bankroll_buyins`) прочитаны как `object` из-за текстовых токенов пропуска и приводятся к числам в задании 1.5.

Признаки `final_bankroll_bi`, `went_broke` и `profitable` производны от `realized_roi`, поэтому при обучении модели на один из этих таргетов остальные нужно исключать из факторов.

In [5]:
FEATURE_KIND = {
    "player_id": "категориальный (идентификатор)", "skill": "количественный (непрерывный)",
    "bankroll_mgmt": "количественный (непрерывный)", "tilt_control": "количественный (непрерывный)",
    "study_hours_week": "количественный (непрерывный)",
    "tournaments_played": "количественный (дискретный)",
    "avg_field_size": "количественный (дискретный)", "avg_buyin": "количественный (непрерывный)",
    "bankroll_buyins": "количественный (дискретный)", "itm_rate": "количественный (доля, %)",
    "final_table_rate": "количественный (доля, %)", "win_rate": "количественный (доля, %)",
    "biggest_cash_x": "количественный (непрерывный)",
    "final_bankroll_bi": "количественный (непрерывный)",
    "realized_roi": "количественный (целевой, %)", "went_broke": "категориальный (бинарный, целевой)",
    "profitable": "категориальный (бинарный, целевой)",
}

numeric = df_na.drop(columns="player_id")
overview = pd.DataFrame({
    "тип_признака": pd.Series(FEATURE_KIND),
    "dtype": df.dtypes.astype(str),
    "nunique": df.nunique(),
    "пропусков": df_na.isnull().sum(),
    "min": numeric.min(),
    "max": numeric.max(),
})
print(overview.to_string())
print()
print("Аномалии:")
print("  ROI = -100% (полная потеря банкролла):", int((numeric["realized_roi"] == -100).sum()))
print("  final_table_rate > itm_rate:",
      int((numeric["final_table_rate"] > numeric["itm_rate"]).sum()))
print("  банкролл 0 при went_broke = 0:",
      int(((numeric["final_bankroll_bi"] == 0) & (numeric["went_broke"] == 0)).sum()))
print("  полных дубликатов:", int(df.duplicated().sum()),
      "| повторов player_id:", int(df["player_id"].duplicated().sum()))
print("  порядковых признаков и столбцов с датой в наборе нет: даты лежат в series_schedule.csv,")
print("  порядковые признаки создаются в заданиях 2.2 и 2.5")
print("  нереалистичных значений нет: доли в [0, 100], ROI ограничен -100% снизу,")
print("  крупные бай-ины до", round(numeric["avg_buyin"].max(), 2), "- хайроллерский хвост распределения")

                                          тип_признака    dtype  nunique  пропусков    min       max
avg_buyin                 количественный (непрерывный)  float64    14673          0    2.0  2913.450
avg_field_size             количественный (дискретный)    int64     2167          0   20.0  3835.000
bankroll_buyins            количественный (дискретный)   object      236        546    5.0   347.000
bankroll_mgmt             количественный (непрерывный)  float64     1000          0    0.0    10.000
biggest_cash_x            количественный (непрерывный)  float64      822          0    0.0   200.000
final_bankroll_bi         количественный (непрерывный)  float64     1980          0    0.0   527.600
final_table_rate              количественный (доля, %)  float64      660          0    0.0    30.000
itm_rate                      количественный (доля, %)  float64      331          0    0.0    60.000
player_id               категориальный (идентификатор)   object    45000          0    NaN 

  ROI = -100% (полная потеря банкролла): 4741
  final_table_rate > itm_rate: 159
  банкролл 0 при went_broke = 0: 0


  полных дубликатов: 700 | повторов player_id: 700
  порядковых признаков и столбцов с датой в наборе нет: даты лежат в series_schedule.csv,
  порядковые признаки создаются в заданиях 2.2 и 2.5
  нереалистичных значений нет: доли в [0, 100], ROI ограничен -100% снизу,
  крупные бай-ины до 2913.45 - хайроллерский хвост распределения


### Задание 1.5. Исправление типов

Четыре количественных столбца прочитаны как `object` из-за текстовых токенов пропуска — приводим к числам через `pd.to_numeric(errors="coerce")`. Строковую дату в расписании серий переводим в `datetime64` через `pd.to_datetime`, бинарные целевые признаки — в `category`.

In [6]:
print("до:")
print(df[TO_NUMERIC + ["went_broke"]].dtypes)

for col in TO_NUMERIC:
    df[col] = pd.to_numeric(df[col], errors="coerce")
for col in ["went_broke", "profitable"]:
    df[col] = df[col].astype("category")
schedule_str = pd.read_csv(f"{DATA_DIR}/series_schedule.csv")
schedule_str["series_start"] = pd.to_datetime(schedule_str["series_start"], format="%Y-%m-%d")

print()
print("после:")
print(df[TO_NUMERIC + ["went_broke"]].dtypes)
print("series_start:", schedule_str["series_start"].dtype)
print()

rows_before, cols_before_raw = df.shape
missing_before = int(df.isnull().sum().sum())
duplicates_before = int(df.duplicated().sum())
print("перед очисткой: строк", rows_before, "| пропусков", missing_before,
      "| дубликатов", duplicates_before)

до:
skill               object
tilt_control        object
study_hours_week    object
bankroll_buyins     object
went_broke           int64
dtype: object

после:
skill                float64
tilt_control         float64
study_hours_week     float64
bankroll_buyins      float64
went_broke          category
dtype: object
series_start: datetime64[ns]



перед очисткой: строк 45700 | пропусков 5641 | дубликатов 700


## Занятие 2. Очистка данных, фильтрация, группировка

### Задание 2.1. Выявление пропущенных значений

In [7]:
missing = pd.DataFrame({"пропусков": df.isnull().sum(),
                        "доля_%": (df.isnull().mean() * 100).round(2)})
print(missing[missing["пропусков"] > 0].sort_values("пропусков", ascending=False).to_string())
print()
for col, share in (df.isnull().mean() * 100).sort_values(ascending=False).items():
    if share > 0:
        print(f"{col:<20} {share:>5.2f}% {'#' * int(round(share * 4))}")
print()
print("всего пропусков:", int(df.isnull().sum().sum()),
      "| строк хотя бы с одним пропуском:", int(df.isnull().any(axis=1).sum()))

                  пропусков  доля_%
study_hours_week       2663    5.83
tilt_control           1525    3.34
skill                   907    1.98
bankroll_buyins         546    1.19

study_hours_week      5.83% #######################
tilt_control          3.34% #############
skill                 1.98% ########
bankroll_buyins       1.19% #####

всего пропусков: 5641 | строк хотя бы с одним пропуском: 5408


### Задание 2.2. Обработка пропусков тремя способами

Пропуски распределены по четырём количественным столбцам и по производному от `skill` интервалу `skill_band`. Способ выбирается по типу столбца и форме распределения.

**Способ 1. Заполнение модой (`fillna`) — категориальный признак.**
`skill_band` строится из `skill` через `pd.cut` и наследует его пропуски: где `skill` пуст, там интервал не определился. Для категории ни среднее, ни медиана не определены — усреднить «новичка» и «профи» нельзя, поэтому берём моду, самый частый уровень мастерства (`любитель`). Заполнено 907 значений. Альтернатива — отдельная категория «неизвестно», но 2% набора не стоят лишнего уровня в каждой группировке.

**Способ 2. Удаление (`dropna`) — безнадёжные записи.**
Сначала проверка по столбцам: столбцов с долей пропусков выше 50% нет, значит по столбцам не удаляем ничего. По строкам применяем `thresh`: запись с одним пропуском остаётся и лечится дальше, запись с двумя и более удаляется, так как восстанавливать её сразу по нескольким каналам — это придумывать данные, а не заполнять. Удалено 228 строк, 0.50% выборки.

**Способ 3. Заполнение медианой и средним (`fillna`) — количественные признаки.**
Выбор между медианой и средним решает перекос распределения, он печатается в выводе ячейки.

| Столбец | Способ | Значение | Заполнено | Перекос | Почему так |
| --- | --- | --- | --- | --- | --- |
| `study_hours_week` | медиана | 4.4 | 2 479 | +1.49 | длинный правый хвост: гриндеры учатся до 40 часов в неделю, среднее они тянут вверх |
| `bankroll_buyins` | медиана | 32.0 | 488 | +1.70 | правый хвост до 347 бай-инов, медиана устойчива к выбросам |
| `skill` | среднее | 5.01 | 813 | −0.00 | симметричная шкала 0–10, среднее использует всю информацию о распределении |
| `tilt_control` | среднее | 5.01 | 1 400 | −0.01 | то же: перекос около нуля, среднее и медиана почти совпадают |

Значение-заполнитель считается по самому столбцу и только по его непропущенным значениям: `median()` и `mean()` игнорируют `NaN`. `bankroll_buyins` после заполнения приводится к `int64`, так как признак дискретный и измеряется в целых бай-инах.

Цена любого заполнения — падение дисперсии: 2 479 игроков получают одинаковые 4.4 часа. При доле пропусков до 6% это приемлемо; при большей доле корректнее заполнять внутри групп через `groupby().transform()` или восстанавливать модельно.

Итог трёх способов: 554 пропущенных значения удалены вместе с 228 строками, 6 087 заполнены, в наборе остаётся 0 пропусков.

In [8]:
df["skill_band"] = pd.cut(df["skill"], bins=[0, 3, 5, 7, 10],
                          labels=["новичок", "любитель", "регуляр", "профи"],
                          include_lowest=True, ordered=True)
filled_band = int(df["skill_band"].isnull().sum())
df["skill_band"] = df["skill_band"].fillna(df["skill_band"].mode()[0])
print("fillna модой:", filled_band, "в skill_band | мода:", df["skill_band"].mode()[0])

print("столбцы с долей пропусков > 50%:", df.columns[df.isnull().mean() > 0.5].tolist() or "нет")
base = [c for c in df.columns if c != "skill_band"]
rows_before_drop = len(df)
df = df.dropna(subset=base, thresh=len(base) - 1).copy()
rows_dropped_na = rows_before_drop - len(df)
print("dropna: удалено строк с двумя и более пропусками:", rows_dropped_na,
      f"({rows_dropped_na / rows_before_drop * 100:.2f}%) | осталось", len(df))

print()
print("перекос до заполнения:", df[TO_NUMERIC].skew().round(2).to_dict())
filled = {}
for col in ["study_hours_week", "bankroll_buyins"]:
    filled[col] = (int(df[col].isnull().sum()), "медиана", round(df[col].median(), 2))
    df[col] = df[col].fillna(df[col].median())
for col in ["skill", "tilt_control"]:
    filled[col] = (int(df[col].isnull().sum()), "среднее", round(df[col].mean(), 2))
    df[col] = df[col].fillna(df[col].mean())
df["bankroll_buyins"] = df["bankroll_buyins"].round().astype("int64")

print(pd.DataFrame(filled, index=["заполнено", "способ", "значение"]).T.to_string())
print()
print("пропусков осталось:", int(df.isnull().sum().sum()))

fillna модой: 907 в skill_band | мода: любитель
столбцы с долей пропусков > 50%: нет
dropna: удалено строк с двумя и более пропусками: 228 (0.50%) | осталось 45472

перекос до заполнения: {'skill': -0.0, 'tilt_control': -0.01, 'study_hours_week': 1.49, 'bankroll_buyins': 1.7}
                 заполнено   способ значение
study_hours_week      2479  медиана      4.4
bankroll_buyins        488  медиана     32.0
skill                  813  среднее     5.01
tilt_control          1400  среднее     5.01

пропусков осталось: 0


### Задание 2.3. Поиск и удаление дубликатов

In [9]:
print("полных дубликатов:", int(df.duplicated().sum()),
      "| дубликатов по player_id:", int(df["player_id"].duplicated().sum()))
print(df[df.duplicated(keep=False)].sort_values("player_id")
        .head(4)[["player_id", "skill", "tournaments_played", "realized_roi"]].to_string(index=False))

rows_before_dup = len(df)
df = df.drop_duplicates().reset_index(drop=True)
rows_dropped_dup = rows_before_dup - len(df)
print()
print("удалено строк-дубликатов:", rows_dropped_dup, "| осталось строк:", len(df),
      "| уникальных player_id:", df["player_id"].nunique())

полных дубликатов: 695 | дубликатов по player_id: 695


player_id  skill  tournaments_played  realized_roi
PLR000162   6.69                  23         -30.5
PLR000162   6.69                  23         -30.5
PLR000190   6.02                  51          11.1
PLR000190   6.02                  51          11.1

удалено строк-дубликатов: 695 | осталось строк: 44777 | уникальных player_id: 44777


### Задание 2.4. Фильтрация по нескольким условиям и сортировка

In [10]:
selected = df[(df["skill"] >= 6) & (df["bankroll_buyins"] >= 50)
              & (df["tournaments_played"] >= 200)
              & ((df["realized_roi"] > 0) | (df["final_table_rate"] >= 3))]

print("условия: skill >= 6 И банкролл >= 50 бай-инов И объём >= 200 турниров")
print("         И (ROI > 0 ИЛИ доля финальных столов >= 3%)")
print("отобрано игроков:", len(selected), f"({len(selected) / len(df) * 100:.2f}%)")
print("средний ROI:", round(selected["realized_roi"].mean(), 2),
      "против", round(df["realized_roi"].mean(), 2), "по набору")
print()
print(selected.sort_values(["skill_band", "realized_roi"], ascending=[False, False])
      [["player_id", "skill_band", "skill", "tournaments_played", "bankroll_buyins", "realized_roi"]]
      .head(10).to_string(index=False))

условия: skill >= 6 И банкролл >= 50 бай-инов И объём >= 200 турниров
         И (ROI > 0 ИЛИ доля финальных столов >= 3%)
отобрано игроков: 490 (1.09%)
средний ROI: 12.48 против -31.88 по набору

player_id skill_band  skill  tournaments_played  bankroll_buyins  realized_roi
PLR008883      профи   8.83                 259               72          87.8
PLR001836      профи  10.00                 297               74          70.0
PLR008754      профи   7.43                 558               50          68.8
PLR006126      профи   7.28                 204               63          66.9
PLR035111      профи   9.05                 223              150          66.7
PLR029128      профи   7.59                 283               54          65.4
PLR028305      профи   7.88                 265               53          63.5
PLR002035      профи   8.94                 225              132          58.9
PLR028499      профи   9.52                 221               90          57.9
PLR018547    

### Задание 2.5. Создание новых признаков

`stake_level` — категоризация `avg_buyin` через `pd.cut` по границам справочника лимитов, ключ присоединения в задании 3.2; `volume_band` — четыре равнонаполненные группы по объёму сыгранных турниров через `pd.qcut`; `discipline_index` — вычисляемый признак, средняя из дисциплинарных шкал `bankroll_mgmt` и `tilt_control`; `venue` — тип площадки по размеру поля, второй ключ присоединения.

In [11]:
df["stake_level"] = pd.cut(df["avg_buyin"], bins=[0, 25, 100, 500, np.inf],
                           labels=["micro", "low", "mid", "high"], right=False, ordered=True)
df["volume_band"] = pd.qcut(df["tournaments_played"], q=4,
                            labels=["очень малый", "малый", "средний", "большой"])
df["discipline_index"] = ((df["bankroll_mgmt"] + df["tilt_control"]) / 2).round(2)
df["venue"] = pd.cut(df["avg_field_size"], bins=[0, 120, 600, np.inf],
                     labels=["live_series", "hybrid", "online_mtt"], right=False).astype(str)

print(df[["player_id", "avg_buyin", "stake_level", "tournaments_played", "volume_band",
          "bankroll_mgmt", "tilt_control", "discipline_index", "avg_field_size", "venue"]]
      .head(5).to_string(index=False))
print()
for col in ["skill_band", "stake_level", "volume_band", "venue"]:
    print(col, dict(df[col].value_counts().sort_index()))
print()
print("discipline_index:", df["discipline_index"].describe()[["mean", "std", "min", "max"]].round(2).to_dict())
print("признаков в наборе:", df.shape[1])

player_id  avg_buyin stake_level  tournaments_played volume_band  bankroll_mgmt  tilt_control  discipline_index  avg_field_size       venue
PLR044346      16.71       micro                  18 очень малый           8.16          5.04              6.60             554      hybrid
PLR004550      66.35         low                 118     средний           1.37          3.17              2.27             909  online_mtt
PLR026203      41.99         low                  93       малый           2.17          6.79              4.48             559      hybrid
PLR032404       8.32       micro                 123     средний           6.69          4.29              5.49            1194  online_mtt
PLR039131      13.39       micro                 160     средний           4.03          5.63              4.83              91 live_series

skill_band {'новичок': np.int64(6893), 'любитель': np.int64(15924), 'регуляр': np.int64(15065), 'профи': np.int64(6895)}
stake_level {'micro': np.int64(15633),

### Задание 2.6. Группировка и агрегация

In [12]:
print(df.groupby("stake_level", observed=True).agg(
    игроков=("player_id", "count"),
    средний_roi=("realized_roi", "mean"),
    медианный_roi=("realized_roi", "median"),
    разброс_roi=("realized_roi", "std"),
    доля_банкротств=("went_broke", lambda s: s.astype(int).mean() * 100),
    средний_скилл=("skill", "mean"),
).round(2).to_string())
print()
print(df.groupby("skill_band", observed=True).agg(
    игроков=("player_id", "count"),
    средний_roi=("realized_roi", "mean"),
    доля_прибыльных=("profitable", lambda s: s.astype(int).mean() * 100),
    средний_discipline=("discipline_index", "mean"),
).round(2).to_string())

             игроков  средний_roi  медианный_roi  разброс_roi  доля_банкротств  средний_скилл
stake_level                                                                                  
micro          15633       -32.40          -34.2        45.51            50.88           4.99
low            22000       -31.60          -33.8        46.00            49.49           5.02
mid             6965       -31.53          -34.1        47.07            49.99           5.00
high             179       -35.23          -35.9        43.37            53.07           5.18



            игроков  средний_roi  доля_прибыльных  средний_discipline
skill_band                                                           
новичок        6893       -47.69             7.49                5.01
любитель      15924       -36.81            14.46                4.99
регуляр       15065       -27.28            22.59                5.00
профи          6895       -14.76            35.63                5.02


### Задание 2.7. Сводная таблица

In [13]:
pivot = pd.pivot_table(df, index="skill_band", columns="volume_band", values="realized_roi",
                       aggfunc="mean", observed=True, margins=True, margins_name="всего").round(2)
print("средний ROI (%):")
print(pivot.to_string())
print()
print("доля банкротств (%):")
print(pd.pivot_table(df, index="skill_band", columns="volume_band", values="went_broke",
                     aggfunc=lambda s: s.astype(int).mean() * 100, observed=True).round(2).to_string())
print()
cells_only = pivot.drop(index="всего", columns="всего").stack(future_stack=True)
print("максимум:", cells_only.idxmax(), cells_only.max(),
      "| минимум:", cells_only.idxmin(), cells_only.min())
print("разброс ROI по объёму:")
print(df.groupby("volume_band", observed=True)["realized_roi"].std().round(2).to_string())

средний ROI (%):
volume_band  очень малый  малый  средний  большой  всего
skill_band                                              
новичок           -40.43 -48.24   -50.42   -51.81 -47.69
любитель          -29.41 -36.46   -39.88   -41.85 -36.81
регуляр           -19.84 -26.82   -29.89   -32.46 -27.28
профи             -10.69 -12.69   -16.00   -19.39 -14.76
всего             -25.17 -31.34   -34.48   -36.59 -31.88

доля банкротств (%):


volume_band  очень малый  малый  средний  большой
skill_band                                       
новичок            20.65  56.75    79.15    93.99
любитель           16.26  46.39    66.32    85.36
регуляр            16.83  40.97    56.76    74.12
профи              14.17  30.48    43.57    57.64

максимум: ('профи', 'очень малый') -10.69 | минимум: ('новичок', 'большой') -51.81
разброс ROI по объёму:


volume_band
очень малый    63.94
малый          42.03
средний        37.12
большой        33.80


**Вывод по сводной таблице.** Наибольший средний ROI — у профи с очень малым объёмом (−10.69%), наименьший — у новичков с большим объёмом (−51.81%). По уровню мастерства порядок ожидаемый: от новичков к профи ROI растёт примерно на 30 п.п. в каждом столбце. Неочевидно другое: внутри любого уровня мастерства ROI падает с ростом объёма, а доля банкротств растёт — у новичков с 20.65% до 93.99%. Это survivorship bias: выплаты в турнирах сильно смещены вверх, поэтому на короткой дистанции часть игроков случайно попадает в крупный занос и тянет среднее группы вверх, а разброс ROI там максимален (63.94 против 33.80 п.п. на большом объёме). С ростом объёма дисперсия вымывается, и результат сходится к истинному ожиданию, отрицательному почти для всех из-за рейка 5–9% от бай-ина.

## Занятие 3. Объединение данных и итоговый EDA-отчёт

### Задание 3.1. `concat` и `merge` с разными `how`

In [14]:
keys = sorted(df["player_id"])
half = len(keys) // 2
top, bottom = df[df["player_id"].isin(keys[:half])], df[df["player_id"].isin(keys[half:])]

vertical = pd.concat([top, bottom], axis=0, ignore_index=True)
print("concat axis=0:", top.shape, "+", bottom.shape, "->", vertical.shape)

horizontal = pd.concat([df[["player_id", "skill", "discipline_index"]],
                        df[["realized_roi", "went_broke"]]], axis=1)
print("concat axis=1:", horizontal.shape)
print(horizontal.head(3).to_string(index=False))
print()

left = df.loc[df["player_id"].isin(keys[:25000]), ["player_id", "skill", "discipline_index"]]
right = df.loc[df["player_id"].isin(keys[20000:]), ["player_id", "realized_roi", "went_broke"]]
print("left:", left.shape, "| right:", right.shape,
      "| общих ключей:", len(set(left["player_id"]) & set(right["player_id"])))
print()
report = []
for how in ["inner", "left", "right", "outer"]:
    merged = pd.merge(left, right, on="player_id", how=how)
    report.append({"how": how, "строк": len(merged),
                   "пропусков_realized_roi": int(merged["realized_roi"].isnull().sum()),
                   "пропусков_skill": int(merged["skill"].isnull().sum())})
print(pd.DataFrame(report).to_string(index=False))

concat axis=0: (22388, 22) + (22389, 22) -> (44777, 22)
concat axis=1: (44777, 5)
player_id  skill  discipline_index  realized_roi went_broke
PLR044346   4.67              6.60         -54.2          1
PLR004550   5.14              2.27         -82.1          1
PLR026203   8.90              4.48         -79.4          1

left: (25000, 3) | right: (24777, 3) | общих ключей: 5000



  how  строк  пропусков_realized_roi  пропусков_skill
inner   5000                       0                0
 left  25000                   20000                0
right  24777                       0            19777
outer  44777                   20000            19777


**Разница в количестве строк.** `inner` оставляет пересечение ключей — 5 000 строк без пропусков. `left` сохраняет все 25 000 ключей левой таблицы, и у 20 000 из них результаты пусты. `right` даёт 24 777 ключей правой таблицы, из них 19 777 без профиля. `outer` берёт объединение — все 44 777 игроков, пропуски появляются с обеих сторон. Ключ `player_id` уникален, поэтому размножения строк нет; при неуникальном ключе получилось бы декартово произведение совпадающих групп. `concat(axis=0)` ключи не сопоставляет вообще: он дописывает строки.

### Задание 3.2. Присоединение тематически связанных справочников

`stake_levels.xlsx` присоединяем по ключу `stake_level`, `venues.json` — по ключу `venue`. Оба соединения `how="left"`: справочники описывают все лимиты и все площадки, поэтому строки набора теряться не должны.

In [15]:
cols_before_merge = df.shape[1]
rows_before_merge = len(df)
df = df.merge(stake_levels, on="stake_level", how="left").merge(venues, on="venue", how="left")
df["stake_level"] = pd.Categorical(df["stake_level"], ["micro", "low", "mid", "high"], ordered=True)

print("строк:", rows_before_merge, "->", len(df), "| столбцов:", cols_before_merge, "->", df.shape[1])
print("пропусков после merge:",
      int(df[["rake_pct", "field_softness", "min_buyin", "payout_top_pct"]].isnull().sum().sum()))
print()
print(df[["player_id", "avg_buyin", "stake_level", "rake_pct", "field_softness",
          "venue", "min_buyin", "payout_top_pct"]].head(5).to_string(index=False))
print()
print(df.groupby(["venue", "stake_level"], observed=True).agg(
    игроков=("player_id", "count"),
    средний_roi=("realized_roi", "mean"),
    рейк=("rake_pct", "first"),
    мягкость_поля=("field_softness", "first"),
    доля_призовых=("payout_top_pct", "first"),
).round(2).to_string())

строк: 44777 -> 44777 | столбцов: 22 -> 29
пропусков после merge: 0

player_id  avg_buyin stake_level  rake_pct  field_softness       venue  min_buyin  payout_top_pct
PLR044346      16.71       micro       9.0               9      hybrid       33.0            13.5
PLR004550      66.35         low       8.0               7  online_mtt        1.1            12.0
PLR026203      41.99         low       8.0               7      hybrid       33.0            13.5
PLR032404       8.32       micro       9.0               9  online_mtt        1.1            12.0
PLR039131      13.39       micro       9.0               9 live_series      110.0            15.0



                         игроков  средний_roi  рейк  мягкость_поля  доля_призовых
venue       stake_level                                                          
hybrid      micro           8863       -31.71   9.0              9           13.5
            low            12346       -30.81   8.0              7           13.5
            mid             3958       -31.01   6.5              5           13.5
            high             103       -37.67   5.0              3           13.5
live_series micro           1382       -32.52   9.0              9           15.0
            low             2037       -31.26   8.0              7           15.0
            mid              587       -29.24   6.5              5           15.0
            high              20       -37.42   5.0              3           15.0
online_mtt  micro           5388       -33.50   9.0              9           12.0
            low             7617       -32.97   8.0              7           12.0
            mid 

## Задание 3.3. Итоговый отчёт разведочного анализа данных

### Краткое описание датасета

Набор Poker Tournament Survivorship (Kaggle, автор sergionefedov) описывает выборку из 45 000 игроков в турнирный покер. Для каждого зафиксированы характеристики подготовки и дисциплины (`skill`, `bankroll_mgmt`, `tilt_control`, `study_hours_week`), параметры игровой практики (`tournaments_played`, `avg_field_size`, `avg_buyin`, `bankroll_buyins`) и результаты по дистанции (`itm_rate`, `final_table_rate`, `win_rate`, `biggest_cash_x`, `final_bankroll_bi`), а также три целевых признака: непрерывный `realized_roi` и бинарные `went_broke`, `profitable`. Рабочая выгрузка `poker_raw.csv` содержала 45 700 строк и 17 столбцов с токенами пропуска и 700 полными дубликатами. Смысловой вопрос набора — survivorship bias: почему результат на коротком отрезке объясняется дисперсией, а не мастерством. После очистки осталось 44 777 строк, после создания признаков и присоединения двух справочников — 29 столбцов.

In [16]:
print(pd.DataFrame({
    "показатель": ["строк", "столбцов", "пропусков", "полных дубликатов",
                   "количественных столбцов типа object"],
    "до_очистки": [rows_before, cols_before_raw, missing_before, duplicates_before, len(TO_NUMERIC)],
    "после_очистки": [len(df), df.shape[1], int(df.isnull().sum().sum()),
                      int(df.duplicated().sum()), 0],
}).to_string(index=False))
print()
print("удалено строк:", rows_before - len(df),
      f"({(rows_before - len(df)) / rows_before * 100:.2f}%) =", rows_dropped_na,
      "с двумя и более пропусками +", rows_dropped_dup, "дубликатов")
print("заполнено значений:", filled_band + sum(v[0] for v in filled.values()))

                         показатель  до_очистки  после_очистки
                              строк       45700          44777
                           столбцов          17             29
                          пропусков        5641              0
                  полных дубликатов         700              0
количественных столбцов типа object           4              0

удалено строк: 923 (2.02%) = 228 с двумя и более пропусками + 695 дубликатов
заполнено значений: 6087


### Инсайт 1. Объём не повышает ROI, а вымывает дисперсию

Разброс ROI падает почти вдвое с ростом объёма: 63.94 п.п. в группе очень малого объёма против 33.80 в группе большого. Средний ROI при этом снижается в каждой группе мастерства (по набору с −25.17% до −36.59%), а доля банкротств растёт с 16.83% до 78.43%. Разрыв между профи и новичками остаётся стабильным: 29.7 п.п. при очень малом объёме и 32.4 п.п. при большом. Значит объём не улучшает результат, а делает его достоверным: короткая дистанция завышает средние за счёт редких крупных заносов, а на длинной остаётся истинное ожидание, отрицательное из-за рейка.

In [17]:
print(pd.pivot_table(df, index="skill_band", columns="volume_band", values="realized_roi",
                     aggfunc="mean", observed=True).round(2).to_string())
print()
print(df.groupby("volume_band", observed=True).agg(
    игроков=("player_id", "count"),
    медианный_объём=("tournaments_played", "median"),
    средний_roi=("realized_roi", "mean"),
    разброс_roi=("realized_roi", "std"),
    доля_банкротств=("went_broke", lambda s: s.astype(int).mean() * 100),
).round(2).to_string())

volume_band  очень малый  малый  средний  большой
skill_band                                       
новичок           -40.43 -48.24   -50.42   -51.81
любитель          -29.41 -36.46   -39.88   -41.85
регуляр           -19.84 -26.82   -29.89   -32.46
профи             -10.69 -12.69   -16.00   -19.39

             игроков  медианный_объём  средний_roi  разброс_roi  доля_банкротств
volume_band                                                                     
очень малый    11227             24.0       -25.17        63.94            16.83
малый          11254             77.0       -31.34        42.03            43.68
средний        11132            154.0       -34.48        37.12            61.60
большой        11164            317.0       -36.59        33.80            78.43


### Инсайт 2. Банкротство определяется управлением банкроллом, а не мастерством

Доля игроков, обнуливших банкролл, падает с 78.73% при банкролле меньше 20 бай-инов до 8.30% при банкролле свыше 100, тогда как по уровню мастерства она меняется куда слабее — с 62.41% у новичков до 36.82% у профи. Это подтверждает и корреляция с `went_broke`: −0.417 у `bankroll_buyins` против −0.158 у `skill` и −0.145 у `discipline_index`. Risk of ruin определяется размером банкролла относительно выбранного лимита, а не силой игры; второй по значимости фактор — объём (+0.398): чем больше сыграно турниров, тем выше шанс, что просадка хотя бы один раз дойдёт до нуля.

In [18]:
print(df.groupby(pd.cut(df["bankroll_buyins"], [0, 20, 50, 100, np.inf],
                        labels=["<20 BI", "20-50 BI", "50-100 BI", ">100 BI"], right=False),
                 observed=True).agg(
    игроков=("player_id", "count"),
    доля_банкротств=("went_broke", lambda s: s.astype(int).mean() * 100),
    средний_roi=("realized_roi", "mean"),
).round(2).to_string())
print()
print(df.groupby("skill_band", observed=True)["went_broke"]
      .apply(lambda s: round(s.astype(int).mean() * 100, 2)).to_string())
print()
print(df[["bankroll_buyins", "discipline_index", "skill", "tournaments_played"]]
      .corrwith(df["went_broke"].astype(int)).round(3).to_string())

                 игроков  доля_банкротств  средний_roi
bankroll_buyins                                       
<20 BI             12606            78.73       -47.21
20-50 BI           19380            49.90       -28.28
50-100 BI          10574            24.96       -22.59
>100 BI             2217             8.30       -20.53

skill_band
новичок     62.41
любитель    53.11
регуляр     47.26
профи       36.82

bankroll_buyins      -0.417
discipline_index     -0.145
skill                -0.158
tournaments_played    0.398


### Инсайт 3. Лимит и площадка меняют дисперсию, а не прибыль

Средний ROI почти не зависит от лимита: от −31.53% на mid до −32.40% на micro, и только хайроллерская группа заметно хуже (−35.23%), но в ней всего 179 игроков. Корреляция ROI с бай-ином равна 0.000, с размером поля — −0.028. Меняется другое: на больших онлайн-полях разброс ROI максимален (49.48 п.п. против 41.57 на лайв-сериях), там же выше доля банкротств (51.62% против 49.48%). Реальный сигнал о силе игры даёт не лимит и не площадка, а сам `skill`: его корреляция с ROI 0.232, то есть на порядок выше любой характеристики площадки.

In [19]:
print(df.groupby("venue", observed=True).agg(
    игроков=("player_id", "count"),
    средний_roi=("realized_roi", "mean"),
    разброс_roi=("realized_roi", "std"),
    доля_банкротств=("went_broke", lambda s: s.astype(int).mean() * 100),
    мягкость_поля=("field_softness", "mean"),
).round(2).to_string())
print()
print(df.groupby("stake_level", observed=True).agg(
    игроков=("player_id", "count"),
    средний_roi=("realized_roi", "mean"),
    разброс_roi=("realized_roi", "std"),
).round(2).to_string())
print()
print(df[["avg_buyin", "avg_field_size", "discipline_index", "skill"]]
      .corrwith(df["realized_roi"]).round(3).to_string())

             игроков  средний_roi  разброс_roi  доля_банкротств  мягкость_поля
venue                                                                         
hybrid         25270       -31.18        44.40            49.20           7.37
live_series     4026       -31.43        41.57            49.48           7.38
online_mtt     15481       -33.14        49.48            51.62           7.37

             игроков  средний_roi  разброс_roi
stake_level                                   
micro          15633       -32.40        45.51
low            22000       -31.60        46.00
mid             6965       -31.53        47.07
high             179       -35.23        43.37

avg_buyin           0.000
avg_field_size     -0.028
discipline_index    0.102
skill               0.232


### Созданные признаки

| Признак | Как получен | Назначение |
| --- | --- | --- |
| `skill_band` | `pd.cut` по `skill`, границы 0/3/5/7/10 | порядковая ось сравнений по уровню мастерства |
| `stake_level` | `pd.cut` по `avg_buyin` по границам справочника лимитов | ключ присоединения `stake_levels.xlsx` |
| `volume_band` | `pd.qcut` по `tournaments_played`, 4 равные группы | измерение дистанции, по нему расходится ROI |
| `discipline_index` | среднее `bankroll_mgmt` и `tilt_control` | единая шкала дисциплины вместо двух коррелирующих признаков |
| `venue` | `pd.cut` по `avg_field_size` | ключ присоединения `venues.json`, прокси типа площадки |

### Общий вывод о пригодности датасета

После очистки набор полон: пропусков и дубликатов нет, все количественные признаки приведены к числовым типам, потеря составила 2.02% строк и пришлась на дубликаты и записи с несколькими пропусками сразу. Объём около 45 тысяч записей, наличие числовых, категориальных и порядковых признаков и трёх согласованных целевых переменных делают набор пригодным и для регрессии по `realized_roi`, и для бинарной классификации по `went_broke` и `profitable`. Главное ограничение — сильная зависимость целевых метрик от объёма сыгранных турниров: `tournaments_played` обязательно учитывать как контрольную переменную, иначе эффект мастерства смешается с дисперсией короткой дистанции; кроме того `final_bankroll_bi`, `went_broke` и `profitable` производны от `realized_roi`, поэтому при обучении на один из этих таргетов остальные нужно исключать из признаков.

## Контрольные вопросы

**1. В чём разница между `Series` и `DataFrame`?**
`Series` — одномерный маркированный массив значений одного типа с индексом. `DataFrame` — двумерная таблица, набор `Series` с общим индексом строк, где каждый столбец имеет свой тип. `df["realized_roi"]` возвращает `Series`, `df[["realized_roi"]]` — `DataFrame` из одного столбца.

**2. Какие есть способы обработки пропущенных значений и когда какой уместен?**
`dropna` — когда доля пропусков критична или в строке пропущено сразу несколько признаков. `fillna` константой или модой — для категориальных признаков и когда сам факт пропуска осмыслен. Среднее — для симметричных числовых распределений, медиана — для перекошенных и с выбросами. Для упорядоченных данных применимы `ffill`/`bfill` и интерполяция, для связанных признаков — заполнение внутри групп через `groupby().transform()`.

**3. Чем отличаются `merge`, `join` и `concat`?**
`concat` склеивает объекты по оси, выравнивая по индексу, без логики ключей. `merge` — реляционное соединение по значениям столбцов с типом `inner`/`left`/`right`/`outer`. `join` — упрощённая форма `merge` по индексу с `how="left"` по умолчанию.

**4. Что произойдёт с количеством строк при `how='inner'` против `how='outer'`, если ключи совпадают не полностью?**
`inner` оставит только пересечение ключей, `outer` — объединение, добавив `NaN` в столбцы без пары. В задании 3.1 `inner` дал 5 000 строк, `outer` — все уникальные ключи обеих таблиц. При неуникальном ключе число строк умножается: получается декартово произведение совпадающих групп.

**5. Для чего нужен `groupby` и как он связан с идеей «split-apply-combine»?**
`groupby` разбивает набор на группы по значениям ключа (split), применяет к каждой агрегацию или преобразование (apply) и собирает результат в один объект с групповым индексом (combine). Это даёт сводные показатели по категориям без явных циклов.

**6. В чём разница между `df.copy()` и обычным присваиванием?**
`df2 = df` создаёт вторую ссылку на тот же объект, и изменения через `df2` видны в `df`. `df.copy()` делает независимую копию данных. Поэтому после фильтрации пишут `.copy()`: иначе результат может быть представлением, и запись в него вызывает `SettingWithCopyWarning`.